# Motion tracking under simulated deformation

Use frozen Farneback optical flow and phase correlation features to fit a regularized model for centroid displacement. Compare against no motion and median foreground flow. The first frame's ground-truth mask supplies foreground pixels. This is pairwise foreground-centroid prediction with a known starting mask. Render objects with changing position, shape and appearance on randomized backgrounds. Hold out entire background groups. The dataset passed filters for 1152 pairs from 72 groups.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'deformation-motion-tracking-heavy-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scikit-learn==1.7.2', 'joblib==1.6.0', 'opencv-python-headless==5.1.0.95', 'Pillow==12.3.0']],check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom sklearn.metrics import (\n    average_precision_score,\n    roc_auc_score,\n)\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", ROOT / "data"))\nSEEDS = [13, 37, 61]\n\n\ndef save(obj, name):\n    folder = ROOT / "checkpoints"\n    folder.mkdir(exist_ok=True)\n    joblib.dump(obj, folder / (name + ".joblib"), compress=3)\n\n\ndef scores(y, p):\n    return {\n        "auroc": float(roc_auc_score(y, p)),\n        "average_precision": float(average_precision_score(y, p)),\n    }\n\n\nimport cv2\nfrom sklearn.linear_model import Ridge\nfrom sklearn.model_selection import GroupShuffleSplit\n\ncv2.setNumThreads(1)\n\n\ndef center(mask):\n    y, x = np.where(mask)\n    return np.array([x.mean(), y.mean()])\n\n\ndef features(a, b, mask):\n    flow = cv2.calcOpticalFlowFarneback(a, b, None, 0.5, 3, 19, 3, 5, 1.2, 0)\n    inside = flow[mask]\n    outside = flow[~mask]\n    local = np.median(inside, axis=0)\n    shift, _ = cv2.phaseCorrelate(a.astype("float32"), b.astype("float32"))\n    return np.r_[\n        local,\n        inside.mean(0),\n        np.quantile(inside, 0.25, axis=0),\n        np.quantile(inside, 0.75, axis=0),\n        np.median(outside, axis=0),\n        shift,\n    ]\n\n\ndef example(rows, a, b, m, n, group):\n    if 30 <= m.sum() < m.size * 0.9 and n.sum() >= 30:\n        rows.append((features(a, b, m), center(n) - center(m), group))\n\n\ndef synthetic():\n    rng = np.random.default_rng(83)\n    rows = []\n    yy, xx = np.mgrid[:96, :128]\n    for group in range(72):\n        background = (\n            cv2.GaussianBlur(rng.normal(0, 17, xx.shape).astype("float32"), (5, 5), 0)\n            + 75\n        )\n        for _ in range(16):\n            x, y = rng.uniform([30, 27], [97, 68])\n            dx, dy = rng.normal(0, 4, 2)\n            rx, ry = rng.uniform(10, 24, 2)\n            masks = [\n                ((xx - x - u) / r) ** 2 + ((yy - y - v) / s) ** 2 < 1\n                for u, v, r, s in [\n                    (0, 0, rx, ry),\n                    (\n                        dx,\n                        dy,\n                        rx * rng.uniform(0.62, 1.35),\n                        ry * rng.uniform(0.62, 1.35),\n                    ),\n                ]\n            ]\n            imgs = [\n                np.uint8(\n                    np.clip(\n                        background\n                        + m * (57 + 9 * np.cos((xx - x - u) / 3))\n                        + rng.normal(0, 4, xx.shape),\n                        0,\n                        255,\n                    )\n                )\n                for m, u in zip(masks, [0, dx])\n            ]\n            example(rows, *imgs, *masks, group)\n    return rows\n\n\ndef run(seed):\n    x, y, g = observations\n    train, test = next(\n        GroupShuffleSplit(n_splits=1, test_size=0.28, random_state=seed).split(x, y, g)\n    )\n    assert not set(g[train]) & set(g[test])\n    model = make_pipeline(StandardScaler(), Ridge(alpha=18)).fit(x[train], y[train])\n    save((model, np.unique(g[test])), f"ridge-{seed}")\n    return {\n        method: {\n            "mean_error_px": float(np.linalg.norm(pred - y[test], axis=1).mean()),\n            "median_error_px": float(np.median(np.linalg.norm(pred - y[test], axis=1))),\n            "test_sequences": len(np.unique(g[test])),\n            "test_pairs": len(test),\n        }\n        for method, pred in [\n            ("stationary", np.zeros_like(y[test])),\n            ("median_flow", x[test, :2]),\n            ("learned_fusion", model.predict(x[test])),\n        ]\n    }\n\n\nif __name__ == "__main__":\n    rows = synthetic()\n    x, y, g = zip(*rows)\n    observations = (np.stack(x), np.stack(y), np.array(g))\n    runs = {seed: run(seed) for seed in SEEDS}\n    means = {\n        method: {\n            key: float(np.mean([run[method][key] for run in runs.values()]))\n            for key in metrics\n        }\n        for method, metrics in runs[SEEDS[0]].items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps(\n            {\n                "seeds": SEEDS,\n                "runs": runs,\n                "means": means,\n                "samples": len(g),\n                "sequences": len(set(g)),\n            },\n            indent=2,\n        )\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())